**Before running:** Runtime → Change runtime type → **T4 GPU**. Run cell 1, then cell 2 (about 15 min), then cell 3 to download.

Open this notebook fresh from GitHub (File → Open notebook → GitHub → `246mayank/cuda-sgemm`), or upload it from your PC.

In [ ]:
# 1 — confirm the GPU
!nvidia-smi

In [ ]:
# 2 — build, benchmark v0-v9, tensor-core track, Nsight Compute (about 15 min)
%cd /content
!rm -rf cuda-sgemm && git clone -q https://github.com/246mayank/cuda-sgemm
%cd /content/cuda-sgemm
!git log --oneline -1
import datetime
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d_%H%M%S')
print('== build'); get_ipython().system('make 2>&1 | grep -E "error|spill" | head -40')
print('== benchmark v0-v9'); get_ipython().system('./bench')
print('== tensor-core track'); get_ipython().system('./bench --tc')
print('== SASS')
!cuobjdump -sass bench | gzip > results/sass_{stamp}.txt.gz && ls -la results/sass_{stamp}.txt.gz
print('== ncu A/B/C (v0-v9 at N=2048)')
!ncu --csv --print-units base -k regex:sgemm_v[0-9] --log-file results/ncu_dram_{stamp}.csv --metrics dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum ./bench --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "A exit: $?"
!ncu --csv --print-units base -k regex:sgemm_v[0-9] --log-file results/ncu_mem_{stamp}.csv --metrics l1tex__t_sector_hit_rate.pct,lts__t_sector_hit_rate.pct,l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,l1tex__t_requests_pipe_lsu_mem_global_op_ld.sum,sm__warps_active.avg.pct_of_peak_sustained_active ./bench --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "B exit: $?"
!ncu --csv --print-units base -k regex:sgemm_v[0-9] --log-file results/ncu_smem_{stamp}.csv --metrics l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum,l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_st.sum,smsp__inst_executed_op_shared_ld.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__inst_executed.sum ./bench --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "C exit: $?"
print('== ncu tensor-core kernels at N=2048')
!ncu --query-metrics --chip tu104 2>&1 | grep -iE 'pipe_tensor' > results/ncu_tcmetrics_{stamp}.txt; head -5 results/ncu_tcmetrics_{stamp}.txt
!ncu --csv --print-units base -k regex:tc[01]_ --log-file results/ncu_tcdram_{stamp}.csv --metrics dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum ./bench --tc --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "T1 exit: $?"
!ncu --csv --print-units base -k regex:tc[01]_ --log-file results/ncu_tcpipe_{stamp}.csv --metrics sm__pipe_tensor_cycles_active.avg.pct_of_peak_sustained_active,sm__inst_executed_pipe_tensor.sum ./bench --tc --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "T2 exit: $?"
!ls results/

In [ ]:
# 3 — bring results home
%cd /content/cuda-sgemm
!zip -qr results.zip results
from google.colab import files; files.download("results.zip")

Kaggle fallback: same cells; download results.zip from the Output panel instead of cell 3. Use `make ARCH=sm_60` if the GPU is a P100 (ncu metric names and clock locking may differ).